## Eda 데이터 탐색

## 데이터 로드

- `ann_df`: bbox 한 개당 한 행 (이미지 메타 + 약 정보 + bbox)
- `img_df`: train 이미지 한 장당 한 행 (촬영 정보 + 객체 수)
- `cat_df`: category_id ↔ 약 이름
- `test_files`: test 이미지 파일명 목록

In [1]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
# 압축을 푼 위치에 따라 sprint_ai_project1_data 폴더가 한 단계 더 있을 수 있다
if (DATA_DIR / "sprint_ai_project1_data").exists():
    DATA_DIR = DATA_DIR / "sprint_ai_project1_data"

TRAIN_IMG_DIR = DATA_DIR / "train_images"
TEST_IMG_DIR = DATA_DIR / "test_images"
ANN_DIR = DATA_DIR / "train_annotations"

IMG_EXTS = {".png", ".jpg", ".jpeg"}
# 이미지 단위로 같은 값을 갖는 촬영 정보. 나머지 필드(drug_N, dl_name 등)는 알약마다 다르다
IMAGE_COLS = ["file_name", "width", "height", "back_color", "light_color", "camera_la", "camera_lo", "size"]


def list_images(folder: Path) -> list[Path]:
    """Zone.Identifier 등을 빼고 이미지 파일만 반환한다."""
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)


def load_annotations(ann_dir: Path) -> pd.DataFrame:
    """json 하나 = 이미지 한 장 속 알약 한 종류. 모든 json을 bbox 한 개당 한 행으로 펼친다."""
    rows = []
    for path in sorted(ann_dir.rglob("*.json")):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        images = {img["id"]: img for img in data["images"]}
        for ann in data["annotations"]:
            x, y, w, h = ann["bbox"]
            rows.append({
                **images[ann["image_id"]],
                "category_id": ann["category_id"],
                "x": x, "y": y, "w": w, "h": h,
                "area": ann["area"],
                "json_path": str(path),
            })
    return pd.DataFrame(rows).drop(columns=["id"])


ann_df = load_annotations(ANN_DIR)

# 이미지 단위 테이블: 어노테이션이 없는 이미지도 포함
img_df = pd.DataFrame({"file_name": [p.name for p in list_images(TRAIN_IMG_DIR)]})
img_meta = ann_df[IMAGE_COLS].drop_duplicates("file_name")
img_df = img_df.merge(img_meta, on="file_name", how="left")
img_df["n_objects"] = img_df["file_name"].map(ann_df["file_name"].value_counts()).fillna(0).astype(int)

# 클래스 테이블: category_id ↔ 약 이름
cat_df = (
    ann_df[["category_id", "dl_name", "dl_name_en"]]
    .drop_duplicates("category_id")
    .sort_values("category_id")
    .reset_index(drop=True)
)

test_files = [p.name for p in list_images(TEST_IMG_DIR)]

print("train 이미지:", len(img_df), "/ test 이미지:", len(test_files))
print("bbox 수:", len(ann_df), "/ 클래스 수:", len(cat_df))
print("어노테이션 없는 train 이미지:", (img_df["n_objects"] == 0).sum())
ann_df.head()

train 이미지: 232 / test 이미지: 842
bbox 수: 763 / 클래스 수: 56
어노테이션 없는 train 이미지: 0


,file_name,width,height,imgfile,drug_N,drug_S,back_color,drug_dir,light_color,camera_la,...,mark_code_front,mark_code_back,change_date,category_id,x,y,w,h,area,json_path
0,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,70,...,,,20160825,1900,644,845,189,190,35910,../data/train_annotations/K-001900-016548-0196...
1,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,75,...,,,20160825,1900,148,241,181,181,32761,../data/train_annotations/K-001900-016548-0196...
2,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,K-001900,정상알약,연회색 배경,앞면,주백색,90,...,,,20160825,1900,167,248,184,182,33488,../data/train_annotations/K-001900-016548-0196...
3,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,K-016548,정상알약,연회색 배경,앞면,주백색,70,...,NaN,NaN,NaN,16548,144,799,239,239,57121,../data/train_annotations/K-001900-016548-0196...
4,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,976,1280,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,K-016548,정상알약,연회색 배경,앞면,주백색,75,...,NaN,NaN,NaN,16548,578,222,233,226,52658,../data/train_annotations/K-001900-016548-0196...
